# 4. Decisión de siembra y reserva de agua

Este capítulo usa el pronóstico de lluvia del capítulo 3 y los parámetros de la finca (en `src/config.py`) para calcular:

1. $x_i^*$: hectáreas máximas de cada cultivo $i$ sin que el almacenamiento se seque.
2. El agua de riego que debe reservarse durante el ciclo.

## Formulación

**Necesidad de riego por hectárea** (la lluvia sobre el cultivo ya se descuenta aquí):

$$NR_{i,t} = \frac{10\cdot\max\left(0,\; K_{c,i,t}\,ET_{0,t} - P^{ef}_t\right)}{\eta}\qquad [\text{m}^3/\text{ha}]$$

**Almacenamiento** de capacidad constante $J$ (condiciones normales), que se vacía con el riego y se llena con la lluvia de la captación:

$$S_t = \min\Big(J,\; S_{t-1} + 10\,P_t\,A_c\,C + Q_{ext} - \sum_i NR_{i,t}\,x_i\Big),\qquad S_0 = J$$

**Problema de decisión:**

$$\max\; Z = \sum_i b_i\,x_i \quad \text{s.a.}\quad \sum_i x_i \le H,\qquad S_t \ge 0\;\;\forall t,\qquad x_i \ge 0$$

Para resolverlo como programa lineal, el $\min(J,\cdot)$ se reemplaza por una variable de rebose $V_t \ge 0$ y la cota $S_t \le J$ (ver `src/hidrologia.optimizar_cultivos`).

**Versión al 95 %:** se usa el percentil 5 de la lluvia pronosticada y el percentil 95 de $ET_0$. La garantía es por semana; la probabilidad conjunta sobre todo el ciclo es menor.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as cfg
np.random.seed(cfg.SEED)
plt.rcParams["figure.dpi"] = 110
from src import hidrologia as hy

faltan = (cfg.faltantes(cfg.FINCA, "FINCA") + cfg.faltantes(cfg.LLUVIA_EFECTIVA, "LLUVIA_EFECTIVA")
          + [f for c, d in cfg.CULTIVOS.items() for f in cfg.faltantes(d, f"CULTIVOS['{c}']")])
LISTO = not faltan
print("Parámetros completos." if LISTO else "Parámetros pendientes en src/config.py:\n  " + "\n  ".join(faltan))

Parámetros pendientes en src/config.py:
  FINCA['H']
  FINCA['J']
  FINCA['A_c']
  FINCA['C']
  FINCA['eta']
  LLUVIA_EFECTIVA['alpha']
  LLUVIA_EFECTIVA['P_min']
  CULTIVOS['maiz']['Kc_ini']
  CULTIVOS['maiz']['Kc_mid']
  CULTIVOS['maiz']['Kc_end']
  CULTIVOS['maiz']['L_ini']
  CULTIVOS['maiz']['L_dev']
  CULTIVOS['maiz']['L_mid']
  CULTIVOS['maiz']['L_late']
  CULTIVOS['maiz']['b']


## 4.1 Escenarios semanales de lluvia y $ET_0$

In [2]:
if LISTO:
    df = pd.read_csv(cfg.DATA_PROCESSED / cfg.ARCHIVO_DATOS, parse_dates=["fecha"])
    pron = pd.read_csv(cfg.DATA_PROCESSED / "pronostico_lluvia_semanal.csv")
    g = df[df.punto == cfg.PUNTO_OBJETIVO].set_index("fecha")
    ultima = g.index.max()

    # ET0 semanal: cuantiles climatológicos por semana del año
    et0_sem = g["ET0"].resample("W").sum()
    woy = et0_sem.index.isocalendar().week.clip(upper=52).values
    et0_q = pd.DataFrame({"et0": et0_sem.values, "woy": woy}).groupby("woy")["et0"]
    semanas_woy = [min(52, (ultima + pd.Timedelta(days=7 * s - 3)).isocalendar().week) for s in pron.semana]

    escenarios = {
        "esperado":        {"P": pron["P_q500"].values, "ET0": et0_q.median().loc[semanas_woy].values},
        "conservador_95":  {"P": pron["P_q050"].values, "ET0": et0_q.quantile(0.95).loc[semanas_woy].values},
    }
    pd.DataFrame({"semana": pron.semana, "fuente": pron.fuente,
                  **{f"P_{k}": v["P"] for k, v in escenarios.items()},
                  **{f"ET0_{k}": v["ET0"] for k, v in escenarios.items()}}).head(20).round(1)

## 4.2 Necesidad de riego por cultivo

In [3]:
SEMANA_SIEMBRA = 1   # semana (desde la última fecha con datos) en que se siembra

def matriz_NR(esc):
    T = len(esc["P"])
    pe = hy.lluvia_efectiva(esc["P"], **cfg.LLUVIA_EFECTIVA)
    NR = np.zeros((len(cfg.CULTIVOS), T))
    for i, (nombre, cult) in enumerate(cfg.CULTIVOS.items()):
        kc = hy.kc_semanal(cult)
        ini = SEMANA_SIEMBRA - 1
        fin = min(T, ini + len(kc))
        NR[i, ini:fin] = hy.necesidad_riego(kc[:fin - ini], esc["ET0"][ini:fin], pe[ini:fin], cfg.FINCA["eta"])
    return NR

if LISTO:
    NRs = {k: matriz_NR(e) for k, e in escenarios.items()}
    fig, ax = plt.subplots(figsize=(11, 3.5))
    for k, NR in NRs.items():
        for i, c in enumerate(cfg.CULTIVOS):
            ax.plot(range(1, NR.shape[1] + 1), NR[i], label=f"{c} – {k}")
    ax.set_xlabel("semana"); ax.set_ylabel("m³/ha"); ax.set_title("Necesidad de riego por hectárea"); ax.legend(); plt.show()

## 4.3 Área máxima y reserva de agua

In [4]:
if LISTO:
    F = cfg.FINCA
    b = [cfg.CULTIVOS[c]["b"] for c in cfg.CULTIVOS]
    resumen = []
    for k, esc in escenarios.items():
        NR = NRs[k]
        # Cada cultivo por separado (búsqueda binaria)
        for i, c in enumerate(cfg.CULTIVOS):
            x1 = hy.area_maxima_un_cultivo(NR[i], esc["P"], F["H"], F["J"], F["A_c"], F["C"], F["Q_ext"])
            resumen.append({"escenario": k, "caso": f"solo {c}", "x_ha": round(x1, 2),
                            "agua_riego_m3": round(float(NR[i].sum() * x1))})
        # Todos los cultivos a la vez (programa lineal)
        r = hy.optimizar_cultivos(NR, esc["P"], b, F["H"], F["J"], F["A_c"], F["C"], F["Q_ext"])
        resumen.append({"escenario": k, "caso": "mezcla óptima",
                        "x_ha": dict(zip(cfg.CULTIVOS, r["x"].round(2))),
                        "agua_riego_m3": round(r["agua_riego_total_m3"])})
    pd.DataFrame(resumen)

In [5]:
if LISTO:
    k = "conservador_95"
    r = hy.optimizar_cultivos(NRs[k], escenarios[k]["P"], b, F["H"], F["J"], F["A_c"], F["C"], F["Q_ext"])
    sim = hy.simular_tanque(r["x"], NRs[k], escenarios[k]["P"], F["J"], F["A_c"], F["C"], F["Q_ext"])
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.plot(sim.semana, sim.nivel_m3, marker="o", ms=3, label="nivel S_t")
    ax.axhline(F["J"], ls="--", c="grey", label="capacidad J")
    ax.bar(sim.semana, sim.consumo_m3, alpha=.3, label="riego")
    ax.set_xlabel("semana"); ax.set_ylabel("m³"); ax.legend(); ax.set_title("Almacenamiento en el escenario conservador (95 %)"); plt.show()
    print("Semana más crítica:", int(sim.loc[sim.nivel_m3.idxmin(), "semana"]), "| nivel mínimo:", round(sim.nivel_m3.min()), "m³")

## 4.4 Sensibilidad

Como varios parámetros de la finca son estimaciones, se recalcula el área máxima variando la capacidad $J$ y la eficiencia $\eta$.

In [6]:
if LISTO:
    c0 = list(cfg.CULTIVOS)[0]
    esc = escenarios["conservador_95"]
    filas = []
    for fJ in [0.5, 0.75, 1.0, 1.25, 1.5]:
        for eta in [0.6, 0.75, 0.9]:
            pe = hy.lluvia_efectiva(esc["P"], **cfg.LLUVIA_EFECTIVA)
            kc = hy.kc_semanal(cfg.CULTIVOS[c0]); T = min(len(kc), len(esc["P"]))
            NR = hy.necesidad_riego(kc[:T], esc["ET0"][:T], pe[:T], eta)
            x = hy.area_maxima_un_cultivo(NR, esc["P"][:T], F["H"], F["J"] * fJ, F["A_c"], F["C"], F["Q_ext"])
            filas.append({"J (m³)": F["J"] * fJ, "eta": eta, "x máx (ha)": round(x, 2)})
    pd.DataFrame(filas).pivot(index="J (m³)", columns="eta", values="x máx (ha)")

**Interpretación.** _Completar con los valores de la finca:_ área recomendada por cultivo, agua a reservar, semana crítica y qué parámetro domina el resultado.